# Week 2 — Dataset Extension





| Dataset | Has | Adds | Validation |
|---------|-----|------|------------|
| HumanEval | NL + Python | Java solution | Real HumanEval-X java_test (same 164 problems) |
| MBPP | NL + Python | Java solution + test driver | Model-generated Main.java from Python asserts |
| HumanEval-X | Python + Java | NL description | Double round-trip: NL→PL1 + NL→PL2 |



## 1. Setup

In [ ]:
# 1.1 — Drive mount + project dir
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    _DEFAULT_DIR = '/content/drive/MyDrive/codegen_week1'
except ImportError:
    _DEFAULT_DIR = '/tmp/codegen_week2'

PROJECT_DIR = Path(os.environ.get('CODEGEN_DATA_DIR', _DEFAULT_DIR))
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Project dir:', PROJECT_DIR)

Mounted at /content/drive
Project dir: /content/drive/MyDrive/codegen_week1


In [ ]:
# 1.2 — install JDK 17 + Python deps
import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-q', 'openjdk-17-jdk-headless'], check=False)
!java -version
!pip install -q transformers>=4.40 bitsandbytes>=0.43 accelerate datasets huggingface_hub javalang pandas tqdm
print('Installs done ✅')

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
Installs done ✅


In [ ]:
# 1.3 — config (no config.py required)
import os
from pathlib import Path

SEED        = int(os.environ.get('CODEGEN_SEED',   '13'))
SAMPLE      = int(os.environ.get('CODEGEN_SAMPLE', '20'))  # problems per dataset
MODEL_SIZE  = os.environ.get('CODEGEN_MODEL', '7b')        # '1.5b' for quick test

DATA_DIR    = Path(os.environ.get('CODEGEN_DATA_DIR', str(PROJECT_DIR)))
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    '1.5b': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    '7b':   'Qwen/Qwen2.5-Coder-7B-Instruct',
}
DECODING_GREEDY = {'do_sample': False, 'max_new_tokens': 512}
EXEC_TIMEOUT_S  = 15

print(f'Seed={SEED}  Sample={SAMPLE}  Model={MODEL_SIZE}')
print(f'Results -> {RESULTS_DIR}')

Seed=13  Sample=20  Model=7b
Results -> /content/drive/MyDrive/codegen_week1/results


In [ ]:
# 1.4 — GPU check
import torch, gc
gc.collect()
torch.cuda.empty_cache()
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU.')
p = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info()
print(f'GPU: {p.name}, {total/1e9:.1f} GB total, {free/1e9:.2f} GB free')

GPU: Tesla T4, 15.6 GB total, 15.53 GB free


## 2. Execution Sandbox

In [ ]:
# 2.1 — Python sandbox
import subprocess, sys, tempfile
from pathlib import Path

def run_python(full_program: str, timeout: int = EXEC_TIMEOUT_S) -> dict:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'solution.py'
        path.write_text(full_program, encoding='utf-8')
        try:
            res = subprocess.run(
                [sys.executable, str(path)],
                capture_output=True, text=True, timeout=timeout,
                env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'},
            )
            if res.returncode == 0:
                return {'passed': True, 'error': None}
            err = (res.stderr or res.stdout).strip().splitlines()[-1:]
            return {'passed': False, 'error': '\n'.join(err) or 'nonzero exit'}
        except subprocess.TimeoutExpired:
            return {'passed': False, 'error': f'timeout >{timeout}s'}
        except Exception as e:
            return {'passed': False, 'error': f'runner: {e!r}'}

In [ ]:
# 2.2 — Java sandbox
# -ea enables Java's assert keyword (OFF by default without this flag)
import re as _re

_CLASS_NAME_RE = _re.compile(r'public\s+class\s+(\w+)')
_JAVA_STD_IMPORTS = (
    'import java.util.*;\n'
    'import java.util.stream.*;\n'
    'import java.util.regex.*;\n'
    'import java.lang.*;\n'
    'import java.math.*;\n'
)

def _inject_imports(src: str) -> str:
    head = src.lstrip()
    if head.startswith('package '):
        nl = head.find('\n')
        return head[:nl+1] + _JAVA_STD_IMPORTS + head[nl+1:]
    return _JAVA_STD_IMPORTS + src


def run_java(solution: str, test_source: str, timeout: int = EXEC_TIMEOUT_S) -> dict:
    """Compile solution + test_source, run `java -ea {main_class}`.
    Works for both assert-style tests (week2 generated) and explicit
    throw-AssertionError tests (HumanEval-X format).
    """
    if not solution or not test_source:
        return {'passed': False, 'error': 'missing solution or test block'}
    classes = _CLASS_NAME_RE.findall(test_source)
    main_class = classes[0] if classes else 'Main'
    solution    = _inject_imports(solution)
    test_source = _inject_imports(test_source)
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        (tmp_path / 'Solution.java').write_text(solution, encoding='utf-8')
        (tmp_path / f'{main_class}.java').write_text(test_source, encoding='utf-8')
        try:
            comp = subprocess.run(
                ['javac', '-d', str(tmp_path),
                 str(tmp_path / 'Solution.java'),
                 str(tmp_path / f'{main_class}.java')],
                capture_output=True, text=True, timeout=timeout,
            )
            if comp.returncode != 0:
                return {'passed': False, 'error': 'compile: ' + (comp.stderr or '').strip()[:400]}
            run = subprocess.run(
                ['java', '-ea', '-cp', str(tmp_path), main_class],
                capture_output=True, text=True, timeout=timeout,
            )
            if run.returncode == 0:
                return {'passed': True, 'error': None}
            tail = (run.stderr or run.stdout).strip().splitlines()[-3:]
            return {'passed': False, 'error': '\n'.join(tail) or 'nonzero exit'}
        except subprocess.TimeoutExpired:
            return {'passed': False, 'error': f'timeout >{timeout}s'}
        except Exception as e:
            return {'passed': False, 'error': f'runner: {e!r}'}


def smoke_test_java() -> dict:
    sol = 'public class Solution { public static int add(int a, int b) { return a + b; } }'
    tst = ('public class Main {\n'
           '    public static void main(String[] args) {\n'
           '        assert Solution.add(2, 3) == 5;\n'
           '        assert Solution.add(-1, 1) == 0;\n'
           '    }\n}\n')
    return run_java(sol, tst)


result = smoke_test_java()
assert result['passed'], f'Java sandbox FAILED: {result}'
print('Java sandbox OK:', result)

Java sandbox OK: {'passed': True, 'error': None}


## 3. Model Loading & Generation

In [ ]:
# 3.1 — model loader
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_qwen(size: str):
    model_id = MODELS[size]
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if size == '7b':
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
    model.eval()
    return tokenizer, model


def unload_model(model) -> None:
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('Model helpers defined.')

Model helpers defined.


In [ ]:
# 3.2 — generation + extraction helpers
import re


def generate(tokenizer, model, prompt: str, decoding: dict) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **decoding,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def _strip_fences(text: str, lang_hints: tuple) -> str:
    s = text.strip()
    open_re = re.compile(r'```(?:' + '|'.join(lang_hints) + r')?\s*\n?', re.IGNORECASE)
    m = open_re.search(s)
    if not m:
        return (s + '\n') if s else ''
    body = s[m.end():]
    close = re.search(r'```', body)
    body = body[:close.start()] if close else body
    return (body.rstrip() + '\n') if body.rstrip() else ''


def extract_python_body(text: str) -> str:
    return _strip_fences(text, ('python', 'py'))


def extract_java_body(text: str) -> str:
    return _strip_fences(text, ('java',))


def llm(tokenizer, model, user_prompt: str, max_new_tokens: int = 1024) -> str:
    """Chat-template generation. Week 2 prompts are fully self-contained so we
    pass them as plain user messages (not wrapped in a Python-completion template).
    """
    msgs = [{'role': 'user', 'content': user_prompt}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return generate(tokenizer, model, prompt, {**DECODING_GREEDY, 'max_new_tokens': max_new_tokens})


assert extract_python_body('```python\ndef f(): pass\n```') == 'def f(): pass\n'
assert extract_java_body('```java\npublic class S {}\n```') == 'public class S {}\n'
print('Generation helpers defined.')

Generation helpers defined.


## 4. Dataset Loaders

In [ ]:
# 4.1 — HumanEval-X raw loader (reused by both HEX extension and HumanEval cross-ref)
import gzip, json, random
from huggingface_hub import hf_hub_download


def _sample(rows, n, seed):
    rows = list(rows)
    random.Random(seed).shuffle(rows)
    return rows[:n]


def _hex_split(lang: str) -> list:
    """Download one HumanEval-X language split, bypassing the broken dataset script."""
    REPO = 'THUDM/humaneval-x'
    candidates = [
        f'data/{lang}/data/humaneval.jsonl',
        f'data/{lang}/data/humaneval_{lang}.jsonl.gz',
        f'data/{lang}/humaneval_{lang}.jsonl.gz',
        f'{lang}/data/humaneval_{lang}.jsonl.gz',
        f'data/{lang}/data/humaneval.jsonl.gz',
    ]
    last_err = None
    for filename in candidates:
        try:
            path = hf_hub_download(repo_id=REPO, filename=filename, repo_type='dataset')
            opener = gzip.open if filename.endswith('.gz') else open
            with opener(path, 'rt', encoding='utf-8') as f:
                return [json.loads(line) for line in f if line.strip()]
        except Exception as e:
            last_err = e
    raise RuntimeError(f'Failed to load HumanEval-X split "{lang}": {last_err!r}')


def load_hex_lookup() -> dict:
    """HumanEval-X rows keyed by problem number ('0', '1', ...).
    Used to attach real java_test + java_declaration to HumanEval rows.
    HumanEval and HumanEval-X share the same 164 problem IDs.
    """
    py_rows = _hex_split('python')
    ja_rows = _hex_split('java')
    by_id_py = {r['task_id'].split('/')[-1]: r for r in py_rows}
    by_id_ja = {r['task_id'].split('/')[-1]: r for r in ja_rows}
    lookup = {}
    for k in by_id_ja:
        if k not in by_id_py:
            continue
        rp, rj = by_id_py[k], by_id_ja[k]
        lookup[k] = {
            'java_test':        rj['test'],
            'java_declaration': rj.get('declaration', rj['prompt']),
            'java_reference':   rj['prompt'] + rj['canonical_solution'],
            'python_test':      rp.get('test'),
            'python_entry_point': rp.get('entry_point'),
        }
    return lookup


print('HumanEval-X loader defined.')

HumanEval-X loader defined.


In [ ]:
# 4.2 — HumanEval, MBPP, and HumanEval-X extension loaders
from datasets import load_dataset


def load_humaneval_for_extension(n: int, seed: int = SEED):
    """HumanEval rows enriched with java_test + java_declaration from HumanEval-X.
    HumanEval/0 <-> HumanEval-X/0 (same problems, different languages).
    This avoids generating a test class — we validate against real Java tests.
    """
    try:
        ds = load_dataset('openai/openai_humaneval', split='test')
    except Exception:
        ds = load_dataset('openai_humaneval', split='test', trust_remote_code=True)

    # load hex lookup once to cross-reference
    hex_lookup = load_hex_lookup()

    rows = []
    for r in ds:
        key = r['task_id'].split('/')[-1]  # 'HumanEval/0' -> '0'
        hx  = hex_lookup.get(key, {})
        rows.append({
            'id':               r['task_id'],
            'nl':               r['prompt'].strip(),
            'python':           r['prompt'] + r['canonical_solution'],
            'python_tests':     r['test'] + f"\ncheck({r['entry_point']})\n",
            # from HumanEval-X: real java test + signature hint
            'java_test':        hx.get('java_test'),       # None if no match
            'java_declaration': hx.get('java_declaration'),
        })
    return _sample(rows, n, seed)


def load_mbpp_for_extension(n: int, seed: int = SEED):
    try:
        ds = load_dataset('google-research-datasets/mbpp', 'sanitized', split='test')
    except Exception:
        ds = load_dataset('mbpp', 'sanitized', split='test', trust_remote_code=True)
    rows = []
    for r in ds:
        rows.append({
            'id':           str(r['task_id']),
            'nl':           r['prompt'],
            'python':       r['code'],
            'python_tests': '\n'.join(r['test_list']),
        })
    return _sample(rows, n, seed)


def load_humaneval_x_for_extension(n: int, seed: int = SEED):
    """HumanEval-X rows. Includes python_test + python_entry_point for NL round-trip validation."""
    py_rows = _hex_split('python')
    ja_rows = _hex_split('java')
    by_id_py = {r['task_id'].split('/')[-1]: r for r in py_rows}
    by_id_ja = {r['task_id'].split('/')[-1]: r for r in ja_rows}
    rows = []
    for k in sorted(by_id_ja.keys(), key=lambda x: int(x) if x.isdigit() else x):
        if k not in by_id_py:
            continue
        rp, rj = by_id_py[k], by_id_ja[k]
        rows.append({
            'id':                  f'hex/{k}',
            'python':              rp['prompt'] + rp['canonical_solution'],
            'java':                rj['prompt'] + rj['canonical_solution'],
            'java_declaration':    rj.get('declaration', rj['prompt']),
            'java_test':           rj['test'],
            # for NL validation round-trip A
            'python_test':         rp.get('test'),
            'python_entry_point':  rp.get('entry_point'),
        })
    return _sample(rows, n, seed)


print('Dataset loaders defined.')

Dataset loaders defined.


## 5. AST Helpers (soft signals — output comparison is the hard gate)

In [ ]:
# 5.1 — Python AST canonicalization + bigrams
import ast
from collections import Counter


class _PyRename(ast.NodeTransformer):
    def __init__(self):
        self.counter = 0
        self.mapping = {}

    def _fresh(self, name):
        if name not in self.mapping:
            self.counter += 1
            self.mapping[name] = f'v{self.counter}'
        return self.mapping[name]

    def visit_FunctionDef(self, node):
        node.args.args = [
            ast.arg(arg=self._fresh(a.arg), annotation=a.annotation)
            for a in node.args.args
        ]
        # strip docstring (ast.Constant only — ast.Str removed in Python 3.12+)
        if (node.body
                and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Constant)
                and isinstance(node.body[0].value.value, str)):
            node.body = node.body[1:] or [ast.Pass()]
        self.generic_visit(node)
        return node

    def visit_Name(self, node):
        if node.id in self.mapping:
            return ast.copy_location(ast.Name(id=self.mapping[node.id], ctx=node.ctx), node)
        return node

    def visit_Assign(self, node):
        for t in node.targets:
            if isinstance(t, ast.Name):
                t.id = self._fresh(t.id)
        self.generic_visit(node)
        return node


def ast_canonicalize_py(src: str) -> str:
    try:
        tree = _PyRename().visit(ast.parse(src))
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except Exception:
        return ''


def _py_bigrams(src: str) -> Counter:
    try:
        nodes = [type(n).__name__ for n in ast.walk(ast.parse(src))]
        return Counter(zip(nodes, nodes[1:]))
    except Exception:
        return Counter()


def python_ast_match(src_a: str, src_b: str) -> tuple:
    """Return (match: bool, error: str|None)."""
    ca, cb = ast_canonicalize_py(src_a), ast_canonicalize_py(src_b)
    if not ca:
        return False, 'code_a: syntax error'
    if not cb:
        return False, 'code_b: syntax error'
    if ca == cb:
        return True, None
    return False, 'python AST mismatch'


# sanity
_a = 'def add(a, b):\n    return a + b'
_b = 'def add(x, y):\n    """doc"""\n    return x + y'
assert ast_canonicalize_py(_a) == ast_canonicalize_py(_b)
print('Python AST helpers OK.')

Python AST helpers OK.


In [ ]:
# 5.2 — Java AST bigrams + cross-language structural jaccard
try:
    import javalang
    _HAS_JAVALANG = True
except Exception:
    _HAS_JAVALANG = False


def _java_text_normalize(src: str) -> str:
    s = re.sub(r'//[^\n]*', '', src)
    s = re.sub(r'/\*.*?\*/', '', s, flags=re.S)
    return re.sub(r'\s+', ' ', s).strip()


def _java_bigrams(src: str) -> Counter:
    if not _HAS_JAVALANG:
        toks = _java_text_normalize(src).split()
        return Counter(zip(toks, toks[1:]))
    try:
        nodes = [type(n).__name__ for _, n in javalang.parse.parse(src)]
        return Counter(zip(nodes, nodes[1:]))
    except Exception:
        toks = _java_text_normalize(src).split()
        return Counter(zip(toks, toks[1:]))


# Normalized vocabulary: map language-specific node type names to shared tokens.
# Both parsers produce 'FUNC', 'IF', 'FOR', 'RETURN' etc. so cross-language
# bigram intersection is non-zero for correct translations (instead of always 0).
_PY_NORM = {
    'FunctionDef': 'FUNC',  'AsyncFunctionDef': 'FUNC',
    'If': 'IF',             'IfExp': 'IF',
    'For': 'FOR',           'AsyncFor': 'FOR',
    'While': 'WHILE',
    'Return': 'RETURN',
    'BinOp': 'BINOP',       'UnaryOp': 'UNOP',
    'BoolOp': 'BOOLOP',     'Compare': 'COMPARE',
    'Call': 'CALL',
    'Assign': 'ASSIGN',     'AugAssign': 'ASSIGN',  'AnnAssign': 'ASSIGN',
    'List': 'LIST',         'Dict': 'DICT',         'Tuple': 'TUPLE',
    'Subscript': 'INDEX',
    'Try': 'TRY',           'ExceptHandler': 'CATCH',
    'ClassDef': 'CLASS',
    'Import': 'IMPORT',     'ImportFrom': 'IMPORT',
    'Constant': 'LITERAL',
    'Name': 'NAME',
    'Attribute': 'ATTR',
    'Module': 'MODULE',
    'Expr': 'STMT',
}
_JAVA_NORM = {
    'MethodDeclaration': 'FUNC',        'ConstructorDeclaration': 'FUNC',
    'IfStatement': 'IF',
    'ForStatement': 'FOR',              'EnhancedForStatement': 'FOR',
    'WhileStatement': 'WHILE',          'DoStatement': 'WHILE',
    'ReturnStatement': 'RETURN',
    'BinaryOperation': 'BINOP',
    'MethodInvocation': 'CALL',
    'Assignment': 'ASSIGN',             'LocalVariableDeclaration': 'ASSIGN',
    'ArrayCreator': 'LIST',             'ArrayInitializer': 'LIST',
    'TryStatement': 'TRY',              'CatchClause': 'CATCH',
    'ClassDeclaration': 'CLASS',
    'Import': 'IMPORT',
    'Literal': 'LITERAL',
    'MemberReference': 'NAME',
    'FieldAccess': 'ATTR',
    'CompilationUnit': 'MODULE',
    'StatementExpression': 'STMT',
}
_JAVA_KW_RE = re.compile(
    r'\b(if|else|for|while|return|new|class|void|int|long|boolean|String'
    r'|null|true|false|throw|try|catch|static|public|private)\b'
)


def _py_norm_bigrams(src: str) -> Counter:
    try:
        nodes = [_PY_NORM.get(type(n).__name__, 'OTHER') for n in ast.walk(ast.parse(src))]
        return Counter(zip(nodes, nodes[1:]))
    except Exception:
        return Counter()


def _java_norm_bigrams(src: str) -> Counter:
    if _HAS_JAVALANG:
        try:
            nodes = [_JAVA_NORM.get(type(n).__name__, 'OTHER') for _, n in javalang.parse.parse(src)]
            return Counter(zip(nodes, nodes[1:]))
        except Exception:
            pass
    kws = _JAVA_KW_RE.findall(src)
    return Counter(zip(kws, kws[1:]))


def ast_jaccard(src_a: str, src_b: str, lang: str) -> float:
    """Same-language jaccard (used for NL round-trip validation)."""
    fn = _py_bigrams if lang == 'python' else _java_bigrams
    A, B = fn(src_a), fn(src_b)
    if not A or not B:
        return 0.0
    inter = sum((A & B).values())
    union = sum((A | B).values())
    return inter / union if union else 0.0


def ast_jaccard_cross(py_src: str, java_src: str) -> float:
    """Cross-language structural jaccard via normalized node vocabulary.
    Python and Java nodes mapped to shared tokens (FUNC/IF/FOR/RETURN/...)
    before bigram comparison. Expect ~0.05-0.30 for correct translations.
    """
    A = _py_norm_bigrams(py_src)
    B = _java_norm_bigrams(java_src)
    if not A or not B:
        return 0.0
    inter = sum((A & B).values())
    union = sum((A | B).values())
    return inter / union if union else 0.0


def ast_canonical_equal(src_a: str, src_b: str, lang: str) -> bool:
    fn = ast_canonicalize_py if lang == 'python' else (lambda s: _java_text_normalize(s))
    a, b = fn(src_a), fn(src_b)
    return bool(a) and bool(b) and a == b


# sanity: cross jaccard must be >0 for equivalent code
_py_ex = 'def add(a, b):\n    return a + b'
_ja_ex = 'public class Solution { public static int add(int a, int b) { return a + b; } }'
_cj = ast_jaccard_cross(_py_ex, _ja_ex)
assert _cj > 0, f'cross jaccard = 0 — normalization broken'
print(f'Java AST helpers OK. javalang={_HAS_JAVALANG}  cross_jaccard_smoke={_cj:.3f} (expect >0)')

Java AST helpers OK. javalang=True  cross_jaccard_smoke=0.294 (expect >0)


## 6. Prompt Builders

In [ ]:
# 6.1 — Python → Java Solution only (Flow A, HumanEval)
# HumanEval has real java_test from HumanEval-X, so we only need Solution.java.
_PY2JAVA_SOL_TMPL = """\
Translate the Python function below to Java.

Return ONE ```java``` block containing:
  - Standard imports at the top (java.util.*, java.util.stream.*, java.math.*)
  - public class Solution with the translated method as public static

Constraints:
  - Class MUST be named Solution. Method MUST be public static.
  - Match the required Java signature exactly.
  - DO NOT import JUnit or any non-stdlib package.
  - Python int -> int or long. bool -> boolean. list -> int[] or List<Integer>. str -> String.
  - Float comparisons: Math.abs(a - b) < 1e-6

# Natural-language description
{nl}

# Python solution (ground truth behavior to match)
```python
{py}
```

# Required Java method signature
```java
{java_decl}
```
"""


def py2java_solution_prompt(nl: str, py: str, java_decl: str) -> str:
    return _PY2JAVA_SOL_TMPL.format(nl=nl, py=py, java_decl=java_decl or '// (no signature hint)')


# 6.2 — Python → Java Solution + test driver (Flow A, MBPP)
# MBPP has no pre-existing Java tests, so we generate both files.
_PY2JAVA_WITH_TEST_TMPL = """\
Translate the Python function below to Java AND write a Main.java test driver.

Return EXACTLY TWO ```java``` blocks: first Solution.java, then Main.java. No prose between.

Constraints:
  - Solution class MUST be named Solution. Method MUST be public static.
  - Main: public static void main(String[] args) using Java assert keyword.
  - DO NOT import JUnit. Only java.util.*, java.util.stream.*, java.math.*.
  - Python int -> int or long. bool -> boolean. list -> int[] or List<Integer>. str -> String.
  - Float comparisons: assert Math.abs(a - b) < 1e-6;
  - Array equality: java.util.Arrays.equals(...)

# Natural-language description
{nl}

# Python solution (ground truth)
```python
{py}
```

# Python tests (re-implement as Java assert statements in Main.main())
```python
{tests}
```

```java
// Solution.java
[full Solution class]
```

```java
// Main.java
[full Main class with assert statements]
```
"""


def py2java_with_test_prompt(nl: str, py: str, tests: str) -> str:
    return _PY2JAVA_WITH_TEST_TMPL.format(nl=nl, py=py, tests=tests)


# 6.3 — Code → NL (Flow B)
_CODE2NL_TMPL = """\
Read the Python and Java implementations of the same function and write a SINGLE-PARAGRAPH
natural-language specification (<=120 words).

Rules:
- Describe the function as if writing a docstring for a new implementer.
- Mention input types, output type, edge cases, and non-obvious behaviour.
- Do NOT name a specific algorithm unless essential.
- Do NOT include code, examples, or prose outside the paragraph.

# Python
```python
{py}
```

# Java
```java
{java}
```

Return only the paragraph, no markdown."""


def code2nl_prompt(py: str, java: str) -> str:
    return _CODE2NL_TMPL.format(py=py, java=java)


# 6.4 — NL → Python (round-trip A)
_NL2PY_TMPL = """\
Implement the following specification in Python.
Return ONLY the function inside a single ```python``` block. No prose, no tests.

# Specification
{nl}

# Required signature
```python
{signature}
```"""


def nl2py_prompt(nl: str, signature: str) -> str:
    return _NL2PY_TMPL.format(nl=nl, signature=signature)


# 6.5 — NL → Java (round-trip B)
_NL2JAVA_TMPL = """\
Implement the following specification in Java.
Return ONLY one ```java``` block: public class Solution with the required public static method.

# Specification
{nl}

# Required Java declaration
```java
{decl}
```"""


def nl2java_prompt(nl: str, decl: str) -> str:
    return _NL2JAVA_TMPL.format(nl=nl, decl=decl)


# 6.6 — feedback prompt (retry)
_FEEDBACK_TMPL = """\
Your previous attempt did not pass validation. Produce a corrected answer in EXACTLY the same format.

# What was wrong
{diff}

# Your previous attempt
{prev}

# Validator output
{err}

# Original task (unchanged)
{original}"""


def feedback_prompt(original: str, prev: str, diff: str, err: str) -> str:
    def cap(s, n=4000): return s if len(s) <= n else s[:n] + '\n...[truncated]'
    return _FEEDBACK_TMPL.format(
        original=cap(original), prev=cap(prev), diff=cap(diff, 1500), err=cap(err, 1500))


print('Prompt builders defined.')

Prompt builders defined.


## 7. Block Extractors, Validators, Differential Test

In [ ]:
# 7.1 — multi-block extractor (for MBPP two-file prompt)
def _extract_blocks(text: str, lang: str):
    pattern = re.compile(r'```\s*' + re.escape(lang) + r'\s*\n(.*?)```', re.S | re.I)
    return [m.group(1).rstrip() for m in pattern.finditer(text)]


def extract_solution_and_test(text: str):
    """Two ```java``` blocks: first = Solution.java, second = Main.java."""
    blocks = _extract_blocks(text, 'java')
    if len(blocks) >= 2:
        return blocks[0], blocks[1]
    if len(blocks) == 1:
        b = blocks[0]
        m = re.search(r'public\s+class\s+Main', b)
        if m:
            return b[:m.start()].rstrip(), b[m.start():]
        return b, ''
    return '', ''


# 7.2 — validators
def validate_java(solution: str, test_src: str) -> dict:
    return run_java(solution, test_src)


# 7.3 — differential Python testing (fallback for NL round-trip A)
def _parse_py_signature(src: str):
    try:
        tree = ast.parse(src)
    except Exception:
        return None, None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            params = []
            for a in node.args.args:
                ann = ''
                if a.annotation is not None:
                    try: ann = ast.unparse(a.annotation)
                    except Exception: pass
                params.append((a.arg, ann.lower()))
            return node.name, params
    return None, None


_TYPE_SAMPLES = {
    'int': [0, 1, 2, -1, 7, 13], 'float': [0.0, 1.5, -2.5, 3.14],
    'bool': [True, False], 'str': ['', 'a', 'abc', 'racecar'],
    'list': [[], [0], [1, 2, 3]], 'tuple': [(), (1,), (1, 2)], 'dict': [{}, {'a': 1}],
}


def _samples_for(hint: str):
    h = (hint or '').lower()
    for key in ('int', 'float', 'bool', 'str', 'list', 'tuple', 'dict'):
        if key in h: return _TYPE_SAMPLES[key]
    return _TYPE_SAMPLES['int']


def differential_python(gt_src: str, gen_src: str, n_inputs: int = 6) -> dict:
    fname, params = _parse_py_signature(gt_src)
    if not fname:
        return {'passed': False, 'error': 'could not parse ground-truth signature'}
    if not gen_src.strip():
        return {'passed': False, 'error': 'empty generated source'}
    import itertools
    combos = list(itertools.islice(
        itertools.product(*[_samples_for(t) for _, t in params]), n_inputs))
    harness = f"""
import sys
{gt_src}
_gt = {fname}
_gen_src = {gen_src!r}
_gen_ns = {{}}
exec(compile(_gen_src, '<gen>', 'exec'), _gen_ns)
_gen = _gen_ns.get({fname!r})
if _gen is None: print('FAIL no_function'); sys.exit(1)
combos = {combos!r}
fails = 0
for c in combos:
    try: a = _gt(*c)
    except Exception as e: a = ('__exc__', type(e).__name__)
    try: b = _gen(*c)
    except Exception as e: b = ('__exc__', type(e).__name__)
    if a != b and not (isinstance(a,float) and isinstance(b,float) and abs(a-b)<1e-6): fails+=1
if fails: print(f'FAIL {{fails}}/{{len(combos)}}'); sys.exit(2)
print(f'OK {{len(combos)}}')
"""
    res = run_python(harness)
    return {'passed': bool(res.get('passed')), 'error': res.get('error') or ''}


# smoke checks
_t = 'Some prose.\n```java\npublic class Solution {}\n```\n```java\npublic class Main {}\n```'
_s, _m = extract_solution_and_test(_t)
assert 'Solution' in _s and 'Main' in _m
print('Validators defined.')

Validators defined.


## 8. Retry Loop

### Retry Loop — How It Works

Three tries max. If it doesn't match, give feedback and try again.

`run_with_retries` is a generic loop used by all three branches:
- **Attempt 1:** Call `generate_fn(None)` — no feedback yet
- **If validation fails:** call `feedback_fn(struct, error)` to build a feedback prompt, then call `generate_fn(feedback)` again
- **Attempt 3:** Last chance — no more retries after this
- **Returns:** `flag=True/False`, best attempt, full history of all tries

Every row is kept regardless of outcome. `flag` tells you which passed.

In [ ]:
def run_with_retries(generate_fn, validate_fn, feedback_fn=None, n_tries: int = 3):
    """
    generate_fn(feedback_or_None) -> (raw_text, struct_dict)
    validate_fn(struct_dict)      -> {passed: bool, error: str}
    feedback_fn(struct, err)      -> feedback passed to next generate_fn
    """
    best, history, feedback = None, [], None
    for attempt in range(1, n_tries + 1):
        raw, struct = generate_fn(feedback)
        verdict = validate_fn(struct)
        history.append({'attempt': attempt, 'verdict': verdict})
        best = {'raw': raw, 'struct': struct, 'verdict': verdict, 'attempt': attempt}
        if verdict.get('passed'):
            return {'flag': True, 'best': best, 'history': history, 'n_attempts': attempt}
        if feedback_fn is None or attempt == n_tries:
            break
        feedback = feedback_fn(struct, verdict.get('error', ''))
    return {'flag': False, 'best': best, 'history': history, 'n_attempts': len(history)}


print('Retry loop defined.')

Retry loop defined.


## 9. Flow Functions

### Flow A-i — HumanEval → +Java

**Why separate from MBPP?** HumanEval and HumanEval-X cover the **exact same 164 problems**.
`load_hex_lookup()` cross-references them by problem number, giving us the **real Java test harness** that was hand-written by the CodeGeeX team.

**What this function does:**
1. Builds a prompt: NL + Python ground truth + Java method signature hint → model generates `Solution.java` (one block only)
2. Validates: `run_java(solution, real_java_test)` — no model-generated test class needed
3. Soft signal: `ast_jaccard_cross(py_gt, java_sol)` — cross-language structural overlap between Python AST bigrams and Java AST bigrams ( *"AST comparison with PL1"*)
4. On failure: passes compiler/runtime error back as feedback, retries up to 3×
5. Rows with no matching HumanEval-X entry are flagged `flag=False, fail_reason='no matching HumanEval-X java_test'` and kept

In [ ]:
# 9.1 — Flow A-i: extend HumanEval with Java
# Uses REAL HumanEval-X java_test — no generated test class needed.
# Validation = output comparison (java_test) + structural jaccard (PL1 vs PL2) as soft signal.

def extend_humaneval_with_java(row, *, tokenizer, model, n_tries: int = 3) -> dict:
    nl       = row['nl']
    py_gt    = row['python']
    java_test_real = row.get('java_test')      # from HumanEval-X cross-ref
    java_decl      = row.get('java_declaration', '')

    if not java_test_real:
        # no matching HumanEval-X entry — mark as skipped rather than crash
        return {
            'dataset': 'humaneval', 'problem_id': row['id'],
            'nl': nl, 'python': py_gt, 'java': '', 'java_test': '',
            'n_attempts': 0, 'flag': False,
            'fail_reason': 'no matching HumanEval-X java_test',
            'ast_jaccard_cross': 0.0, 'last_error': '',
        }

    original_prompt = py2java_solution_prompt(nl, py_gt, java_decl)

    def generate_fn(feedback):
        if feedback is None:
            prompt = original_prompt
        else:
            prompt = feedback_prompt(
                original=original_prompt,
                prev=feedback['prev'], diff=feedback['diff'], err=feedback['err'])
        raw = llm(tokenizer, model, prompt, max_new_tokens=1200)
        sol = extract_java_body(raw)
        return raw, {'java_solution': sol}

    def validate_fn(struct):
        sol = struct.get('java_solution', '')
        if not sol:
            return {'passed': False, 'error': 'empty java solution'}
        return validate_java(sol, java_test_real)

    def feedback_fn(struct, err):
        return {
            'prev': '```java\n' + (struct.get('java_solution') or '') + '\n```',
            'diff': ('Java solution failed the test harness. Fix the method signature, '
                     'types, and logic so it compiles and passes `java -ea Main`.'),
            'err': err,
        }

    res  = run_with_retries(generate_fn, validate_fn, feedback_fn, n_tries=n_tries)
    best = res['best']
    sol  = best['struct'].get('java_solution', '')

    # Soft signal: cross-language structural jaccard (Python AST bigrams vs Java bigrams)
    # This is the 'AST comparison with PL1'
    cross_jaccard = ast_jaccard_cross(py_gt, sol) if sol else 0.0

    return {
        'dataset':           'humaneval',
        'problem_id':        row['id'],
        'nl':                nl,
        'python':            py_gt,
        'java':              sol,
        'java_test':         java_test_real,
        'n_attempts':        res['n_attempts'],
        'flag':              res['flag'],
        'fail_reason':       '' if res['flag'] else (best['verdict'].get('error', '') or 'unknown')[:500],
        'ast_jaccard_cross': round(cross_jaccard, 4),  # PL1 Python vs PL2 Java structural similarity
        'last_error':        (best['verdict'].get('error', '') or '')[:1000],
    }


print('extend_humaneval_with_java defined.')

extend_humaneval_with_java defined.


### Flow A-ii — MBPP → +Java

**Why separate from HumanEval?** MBPP has no equivalent in HumanEval-X, so there are no pre-existing Java tests.

**What this function does:**
1. Builds a prompt asking for **two** `java` blocks: `Solution.java` + `Main.java` test driver
2. `Main.java` is derived from Python asserts — same test cases, translated to Java `assert` statements
3. Validates: `run_java(solution, main)` with `-ea` flag so Java asserts are active
4. Soft signal: same `ast_jaccard_cross` cross-language jaccard as HumanEval
5. Feedback loop includes the full compiler/runtime error from javac/java

This is less rigorous than HumanEval (model writes both the solution AND the tests) but it's the best available option for MBPP since no ground-truth Java exists.

In [ ]:
# 9.2 — Flow A-ii: extend MBPP with Java
# No pre-existing Java tests for MBPP. Model generates both Solution.java + Main.java.
# Main.java assertions are translated from Python asserts (not invented from scratch).

def extend_mbpp_with_java(row, *, tokenizer, model, n_tries: int = 3) -> dict:
    nl       = row['nl']
    py_gt    = row['python']
    py_tests = row['python_tests']
    original_prompt = py2java_with_test_prompt(nl, py_gt, py_tests)

    def generate_fn(feedback):
        if feedback is None:
            prompt = original_prompt
        else:
            prompt = feedback_prompt(
                original=original_prompt,
                prev=feedback['prev'], diff=feedback['diff'], err=feedback['err'])
        raw = llm(tokenizer, model, prompt, max_new_tokens=1400)
        sol, tst = extract_solution_and_test(raw)
        return raw, {'java_solution': sol, 'java_test': tst}

    def validate_fn(struct):
        return validate_java(struct.get('java_solution', ''), struct.get('java_test', ''))

    def feedback_fn(struct, err):
        prev_blob = (
            '```java\n// Solution.java\n' + (struct.get('java_solution') or '') + '\n```\n'
            '```java\n// Main.java\n'     + (struct.get('java_test')     or '') + '\n```'
        )
        return {
            'prev': prev_blob,
            'diff': ('javac + java -ea Main failed. Fix: correct method signature, '
                     'types, static modifier. NO JUnit. Use plain assert keyword.'),
            'err': err,
        }

    res  = run_with_retries(generate_fn, validate_fn, feedback_fn, n_tries=n_tries)
    best = res['best']
    sol  = best['struct'].get('java_solution', '')
    tst  = best['struct'].get('java_test', '')

    cross_jaccard = ast_jaccard_cross(py_gt, sol) if sol else 0.0

    return {
        'dataset':           'mbpp',
        'problem_id':        row['id'],
        'nl':                nl,
        'python':            py_gt,
        'java':              sol,
        'java_test':         tst,
        'n_attempts':        res['n_attempts'],
        'flag':              res['flag'],
        'fail_reason':       '' if res['flag'] else (best['verdict'].get('error', '') or 'unknown')[:500],
        'ast_jaccard_cross': round(cross_jaccard, 4),
        'last_error':        (best['verdict'].get('error', '') or '')[:1000],
    }


print('extend_mbpp_with_java defined.')

extend_mbpp_with_java defined.


### Flow B — HumanEval-X → +NL

**Goal:** Generate one English description that captures both the Python and Java implementations.

**Why double round-trip validation?** *"NL→PL1 and NL→PL2 — if both codes match ground truth in terms of AST and output, the description is good enough."*

**What this function does:**
1. Prompt: both Python + Java ground truth → model writes a single paragraph NL
2. Round-trip A (NL → Python):
   - Regenerate Python from NL
   - If row has `python_test` + `python_entry_point` (from HumanEval-X): run the **real** Python test harness + check AST match
   - Else: fall back to differential testing on synthesized inputs
3. Round-trip B (NL → Java):
   - Regenerate Java from NL
   - Run against **real** `java_test` from HumanEval-X
   - Compute AST jaccard vs Java ground truth
4. **Both A and B must pass.** Failure → improve NL with feedback → retry ≤ 3×

`flag=True` means the NL was precise enough to regenerate both languages correctly.

In [ ]:
# 9.3 — Flow B: extend HumanEval-X with NL
# Validate NL via double round-trip:
#   A: NL -> PL1' -> run real Python test (+ AST match with PL1)
#   B: NL -> PL2' -> run real java_test (+ AST jaccard with PL2)
# Both must pass. Feedback on failure. Max 3 tries.

def _extract_py_sig_str(py_src: str) -> str:
    fname, params = _parse_py_signature(py_src)
    if not fname:
        return ''
    return f"def {fname}({', '.join(p for p, _ in params)}):"


def extend_with_nl(row, *, tokenizer, model, n_tries: int = 3) -> dict:
    py_gt         = row['python']
    java_gt       = row['java']
    java_test_gt  = row['java_test']
    java_decl     = row['java_declaration']
    py_sig        = _extract_py_sig_str(py_gt)
    py_test       = row.get('python_test')        # real Python test from HumanEval-X
    py_entry_point = row.get('python_entry_point')
    original_prompt = code2nl_prompt(py_gt, java_gt)

    def generate_nl(feedback):
        if feedback is None:
            prompt = original_prompt
        else:
            prompt = feedback_prompt(
                original=original_prompt,
                prev=feedback['prev'], diff=feedback['diff'], err=feedback['err'])
        raw = llm(tokenizer, model, prompt, max_new_tokens=400)
        nl = re.sub(r'```[a-zA-Z]*\n.*?```', '', raw, flags=re.S).strip()
        return raw, {'nl': nl}

    def validate_nl(struct):
        nl = struct.get('nl', '')
        if not nl:
            return {'passed': False, 'error': 'empty NL'}

        # --- Round-trip A: NL -> Python ---
        if py_sig:
            py_attempt = llm(tokenizer, model, nl2py_prompt(nl, py_sig), max_new_tokens=600)
            gen_py = extract_python_body(py_attempt)

            if py_test and py_entry_point:
                # Use real Python test from HumanEval-X (output comparison with PL1)
                full_py = gen_py + '\n\n' + py_test + f'\ncheck({py_entry_point})\n'
                py_exec_res = run_python(full_py)
                # Also AST comparison with PL1
                ast_ok, ast_err = python_ast_match(py_gt, gen_py) if gen_py else (False, 'empty')
                py_passed = py_exec_res['passed']
                py_err = py_exec_res.get('error') or ''
                if not ast_ok:
                    py_err += f' | AST: {ast_err}'
            else:
                # Fallback: synthesized input differential test
                diff_res = differential_python(py_gt, gen_py)
                py_passed = diff_res['passed']
                py_err = diff_res.get('error', '')
                ast_ok = False
        else:
            gen_py = ''
            py_passed, py_err, ast_ok = False, 'no python signature', False

        # --- Round-trip B: NL -> Java ---
        java_attempt = llm(tokenizer, model, nl2java_prompt(nl, java_decl), max_new_tokens=900)
        gen_java = extract_java_body(java_attempt)
        java_res = (validate_java(gen_java, java_test_gt)
                    if gen_java else {'passed': False, 'error': 'empty java'})
        java_passed = java_res['passed']

        struct['_gen_py']    = gen_py
        struct['_gen_java']  = gen_java
        struct['_ast_match'] = ast_ok
        struct['_java_res']  = java_res

        passed = py_passed and java_passed
        err = ''
        if not py_passed:   err += f'py round-trip: {py_err[:300]} | '
        if not java_passed: err += f'java round-trip: {java_res.get("error","")[:300]}'
        return {'passed': bool(passed), 'error': err.strip()}

    def feedback_fn(struct, err):
        return {
            'prev': struct.get('nl', ''),
            'diff': ('Round-trip failed: code regenerated from your NL did not match ground truth. '
                     'Be more precise about input/output types, edge cases, exact behaviour. '
                     'Avoid ambiguity that lets a coder pick a different algorithm.'),
            'err': err,
        }

    res  = run_with_retries(generate_nl, validate_nl, feedback_fn, n_tries=n_tries)
    best = res['best']
    nl       = best['struct'].get('nl', '')
    gen_py   = best['struct'].get('_gen_py', '')
    gen_java = best['struct'].get('_gen_java', '')

    return {
        'dataset':               'humaneval_x_py2java',
        'problem_id':            row['id'],
        'nl':                    nl,
        'python':                py_gt,
        'java':                  java_gt,
        'java_test':             java_test_gt,
        'n_attempts':            res['n_attempts'],
        'flag':                  res['flag'],
        'fail_reason':           '' if res['flag'] else (best['verdict'].get('error', '') or 'unknown')[:500],
        'ast_canonical_match_py': best['struct'].get('_ast_match', False),
        'ast_jaccard_py':         ast_jaccard(py_gt, gen_py,   'python') if gen_py   else 0.0,
        'ast_jaccard_java':       ast_jaccard(java_gt, gen_java, 'java') if gen_java else 0.0,
        'last_error':             (best['verdict'].get('error', '') or '')[:1000],
    }


print('extend_with_nl defined.')

extend_with_nl defined.


## 10. Load Datasets

In [ ]:
he_rows   = load_humaneval_for_extension(SAMPLE)
mbpp_rows = load_mbpp_for_extension(SAMPLE)
hex_rows  = load_humaneval_x_for_extension(SAMPLE)

he_with_test = sum(1 for r in he_rows if r.get('java_test'))
print(f'HumanEval rows: {len(he_rows)}  (have java_test from HumanEval-X: {he_with_test})')
print(f'MBPP rows:      {len(mbpp_rows)}')
print(f'HumanEval-X rows: {len(hex_rows)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

humaneval.jsonl:   0%|          | 0.00/343k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/43 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

HumanEval rows: 20  (have java_test from HumanEval-X: 20)
MBPP rows:      20
HumanEval-X rows: 20


## 11. Load Model

In [ ]:
# Change MODEL_SIZE in cell 1.3 to '1.5b' for a quick 2-min sanity run
gc.collect(); torch.cuda.empty_cache()
tokenizer, model = load_qwen(MODEL_SIZE)
free, total = torch.cuda.mem_get_info()
print(f'GPU after load: {free/1e9:.2f} / {total/1e9:.2f} GB free')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU after load: 9.79 / 15.64 GB free


## 12. Run HumanEval Extension (→ +Java using real HumanEval-X tests)

### Run — HumanEval Extension

Calls `extend_humaneval_with_java` for each sampled row. Saves results to:
`results/extended_humaneval_seed{N}.csv`

Key output columns: `flag`, `n_attempts`, `ast_jaccard_cross`, `fail_reason`, `java`, `java_test`

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

records = []
for r in tqdm(he_rows, desc='HumanEval → +Java'):
    records.append(extend_humaneval_with_java(r, tokenizer=tokenizer, model=model))

df_he = pd.DataFrame(records)
out_path = RESULTS_DIR / f'extended_humaneval_seed{SEED}.csv'
df_he.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'flag=True: {(df_he["flag"]==True).sum()} / {len(df_he)}  '
      f'mean_attempts={df_he["n_attempts"].mean():.2f}  '
      f'mean_jaccard_cross={df_he["ast_jaccard_cross"].mean():.3f}')
df_he[['problem_id','flag','n_attempts','ast_jaccard_cross','fail_reason']].tail(5)

HumanEval → +Java:   0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Saved → /content/drive/MyDrive/codegen_week1/results/extended_humaneval_seed13.csv
flag=True: 18 / 20  mean_attempts=1.20  mean_jaccard_cross=0.202


,problem_id,flag,n_attempts,ast_jaccard_cross,fail_reason
15,HumanEval/100,True,1,0.1538,
16,HumanEval/130,True,1,0.1467,
17,HumanEval/125,False,3,0.1414,"Exception in thread ""main"" java.lang.Assertion..."
18,HumanEval/69,True,1,0.3958,
19,HumanEval/12,True,1,0.2500,


## 13. Run MBPP Extension (→ +Java with model-generated test driver)

### Run — MBPP Extension

Calls `extend_mbpp_with_java` for each sampled row. Saves results to:
`results/extended_mbpp_seed{N}.csv`

Same columns as HumanEval. `java_test` here = model-generated `Main.java` (not a ground-truth test).

In [ ]:
records = []
for r in tqdm(mbpp_rows, desc='MBPP → +Java'):
    records.append(extend_mbpp_with_java(r, tokenizer=tokenizer, model=model))

df_mbpp = pd.DataFrame(records)
out_path = RESULTS_DIR / f'extended_mbpp_seed{SEED}.csv'
df_mbpp.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'flag=True: {(df_mbpp["flag"]==True).sum()} / {len(df_mbpp)}  '
      f'mean_attempts={df_mbpp["n_attempts"].mean():.2f}  '
      f'mean_jaccard_cross={df_mbpp["ast_jaccard_cross"].mean():.3f}')
df_mbpp[['problem_id','flag','n_attempts','ast_jaccard_cross','fail_reason']].tail(5)

MBPP → +Java:   0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Saved → /content/drive/MyDrive/codegen_week1/results/extended_mbpp_seed13.csv
flag=True: 19 / 20  mean_attempts=1.25  mean_jaccard_cross=0.231


,problem_id,flag,n_attempts,ast_jaccard_cross,fail_reason
15,140,False,3,0.2326,compile: /tmp/tmph8xb_bxg/Main.java:12: error:...
16,103,True,1,0.1959,
17,268,True,1,0.1724,
18,66,True,1,0.2619,
19,442,True,1,0.2923,


## 14. Run HumanEval-X Extension (→ +NL via double round-trip)

### Run — HumanEval-X Extension

Calls `extend_with_nl` for each sampled row. Saves results to:
`results/extended_humaneval_x_seed{N}.csv`

Key output columns: `flag`, `n_attempts`, `nl`, `ast_canonical_match_py`, `ast_jaccard_py`, `ast_jaccard_java`, `fail_reason`

In [ ]:
records = []
for r in tqdm(hex_rows, desc='HumanEval-X → +NL'):
    records.append(extend_with_nl(r, tokenizer=tokenizer, model=model))

df_hex = pd.DataFrame(records)
out_path = RESULTS_DIR / f'extended_humaneval_x_seed{SEED}.csv'
df_hex.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'flag=True: {(df_hex["flag"]==True).sum()} / {len(df_hex)}  '
      f'mean_attempts={df_hex["n_attempts"].mean():.2f}')
df_hex[['problem_id','flag','n_attempts','ast_jaccard_py','ast_jaccard_java','fail_reason']].tail(5)

HumanEval-X → +NL:   0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Saved → /content/drive/MyDrive/codegen_week1/results/extended_humaneval_x_seed13.csv
flag=True: 14 / 20  mean_attempts=1.70


,problem_id,flag,n_attempts,ast_jaccard_py,ast_jaccard_java,fail_reason
15,hex/100,True,2,0.171875,0.517241,
16,hex/130,False,3,0.741379,0.648352,py round-trip: FAIL 4/6 | java round-trip: \ta...
17,hex/125,False,3,0.555556,0.000000,"java round-trip: Exception in thread ""main"" ja..."
18,hex/69,True,1,0.563218,0.772727,
19,hex/12,True,1,0.485294,1.000000,


## 15. Unload Model & Summary

In [ ]:
unload_model(model)
del tokenizer
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'GPU after unload: {free/1e9:.2f} / {total/1e9:.2f} GB free')

GPU after unload: 9.77 / 15.64 GB free


In [ ]:
def _summary(df, name):
    row = {
        'dataset':       name,
        'n_total':       len(df),
        'n_flag_true':   int((df['flag'] == True).sum()),
        'pct_flag_true': round(100 * (df['flag'] == True).mean(), 1),
        'mean_attempts': round(df['n_attempts'].mean(), 2),
    }
    if 'ast_jaccard_cross' in df.columns:
        row['mean_jaccard_cross'] = round(df['ast_jaccard_cross'].mean(), 3)
    return row

summary = pd.DataFrame([
    _summary(df_he,   'humaneval'),
    _summary(df_mbpp, 'mbpp'),
    _summary(df_hex,  'humaneval_x_py2java'),
])
summary.to_csv(RESULTS_DIR / 'extension_summary.csv', index=False)
print(summary.to_string(index=False))

            dataset  n_total  n_flag_true  pct_flag_true  mean_attempts  mean_jaccard_cross
          humaneval       20           18           90.0           1.20               0.202
               mbpp       20           19           95.0           1.25               0.231
humaneval_x_py2java       20           14           70.0           1.70                 NaN


## Done

**Outputs:**
- `extended_humaneval_seed{N}.csv` — HumanEval + Java (validated against real HumanEval-X tests)
- `extended_mbpp_seed{N}.csv` — MBPP + Java (model-generated test driver from Python asserts)
- `extended_humaneval_x_seed{N}.csv` — HumanEval-X + NL (double round-trip validated)
- `extension_summary.csv` — pass rate + mean attempts + structural jaccard per dataset

**Validation:**
- **Output comparison with PL1**: Java runs against real/derived tests that encode PL1 behavior
- **AST comparison with PL1**: `ast_jaccard_cross` = cross-language structural bigram overlap
- **3-try feedback loop**: `run_with_retries` with validator error in feedback prompt
- **Flag every row**: `flag=True/False` + `fail_reason` + `n_attempts`, no rows dropped

